In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from datetime import datetime

LAKEHOUSE = "default"
BRONZE_TABLE   = "{LAKEHOUSE}.dbo.bronze_bars_v2"
SILVER_DAILY   = "{LAKEHOUSE}.dbo.silver_daily"
SILVER_5MIN    = "{LAKEHOUSE}.dbo.silver_5min"
SILVER_1MIN    = "{LAKEHOUSE}.dbo.silver_1min"

# Timezone for session labelling — US Eastern
ET = "America/New_York"

StatementMeta(, cc821a45-bf34-4744-adea-5258840ae033, 3, Finished, Available, Finished, False)

In [2]:
# ── STEP 1: READ BRONZE ───────────────────────────────────────
bronze = spark.read.format("delta").table(BRONZE_TABLE)
print(f"Bronze rows: {bronze.count():,}")
bronze.printSchema()

StatementMeta(, cc821a45-bf34-4744-adea-5258840ae033, 4, Finished, Available, Finished, False)

Bronze rows: 80,667
root
 |-- ticker: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- sub_industry: string (nullable = true)
 |-- granularity: string (nullable = true)
 |-- t: long (nullable = true)
 |-- o: double (nullable = true)
 |-- h: double (nullable = true)
 |-- l: double (nullable = true)
 |-- c: double (nullable = true)
 |-- v: double (nullable = true)
 |-- vw: double (nullable = true)
 |-- n: long (nullable = true)



In [3]:
# ── STEP 2: RENAME + BASE TRANSFORMS ─────────────────────────
# Applied to all granularities before splitting.
# - Rename single-char Polygon columns to meaningful names
# - Convert Unix ms timestamp to UTC datetime
# - Drop rows with null/zero prices or volume (bad ticks)

def base_transform(df):
    return (
        df
        # ── Rename ────────────────────────────────────────────
        .withColumnRenamed("t",  "timestamp_utc")
        .withColumnRenamed("o",  "open")
        .withColumnRenamed("h",  "high")
        .withColumnRenamed("l",  "low")
        .withColumnRenamed("c",  "close")
        .withColumnRenamed("v",  "volume")
        .withColumnRenamed("vw", "vwap")
        .withColumnRenamed("n",  "transaction_count")

        # ── Convert timestamp (Unix ms → UTC datetime) ────────
        .withColumn("timestamp", F.to_timestamp(F.col("timestamp_utc") / 1000))

        # ── Data quality: drop bad ticks ──────────────────────
        .filter(F.col("open").isNotNull()  & (F.col("open")  > 0))
        .filter(F.col("close").isNotNull() & (F.col("close") > 0))
        .filter(F.col("high").isNotNull()  & (F.col("high")  > 0))
        .filter(F.col("low").isNotNull()   & (F.col("low")   > 0))
        .filter(F.col("volume").isNotNull() & (F.col("volume") > 0))
        # high must be >= low (sanity check)
        .filter(F.col("high") >= F.col("low"))

        # ── Drop raw timestamp_utc (redundant now) ────────────
        .drop("timestamp_utc", "granularity")
    )

print("base_transform() defined")

StatementMeta(, cc821a45-bf34-4744-adea-5258840ae033, 5, Finished, Available, Finished, False)

base_transform() defined


In [4]:
# ── STEP 3: TIME ENRICHMENT ───────────────────────────────────
# Two flavours: daily and intraday (1min / 5min)

def enrich_daily(df):
    """Add calendar-based time columns for daily bars."""
    return (
        df
        .withColumn("date",           F.to_date("timestamp"))
        .withColumn("day_of_week",    F.date_format("timestamp", "EEEE"))       # Monday, Tuesday …
        .withColumn("week_of_year",   F.weekofyear("timestamp").cast("int"))
        .withColumn("month",          F.month("timestamp").cast("int"))
        .withColumn("quarter",        F.concat(F.lit("Q"), F.quarter("timestamp").cast("string")))
        .withColumn("year",           F.year("timestamp").cast("int"))
        .withColumn("is_month_end",   F.col("date") == F.last_day("date"))
        .withColumn("is_quarter_end",
            (F.col("month").isin([3, 6, 9, 12])) & (F.col("date") == F.last_day("date"))
        )
    )


def enrich_intraday(df):
    """Add intraday time columns for 1min and 5min bars."""
    et_ts = F.to_timestamp(
        F.from_utc_timestamp(F.col("timestamp"), ET)
    )
    return (
        df
        .withColumn("timestamp_et",  F.from_utc_timestamp(F.col("timestamp"), ET))
        .withColumn("date",          F.to_date("timestamp_et"))
        .withColumn("hour",          F.hour("timestamp_et").cast("int"))
        .withColumn("minute",        F.minute("timestamp_et").cast("int"))
        .withColumn("day_of_week",   F.date_format("timestamp_et", "EEEE"))
        # session: pre (04:00-09:29), regular (09:30-15:59), after (16:00-19:59)
        .withColumn("time_int",      (F.hour("timestamp_et") * 100 + F.minute("timestamp_et")).cast("int"))
        .withColumn("session",
            F.when(F.col("time_int").between(400,  929),  "pre")
             .when(F.col("time_int").between(930,  1559), "regular")
             .when(F.col("time_int").between(1600, 1959), "after")
             .otherwise("extended")
        )
        .drop("time_int")
    )

print("enrich_daily() and enrich_intraday() defined")

StatementMeta(, cc821a45-bf34-4744-adea-5258840ae033, 6, Finished, Available, Finished, False)

enrich_daily() and enrich_intraday() defined


In [5]:
# ── STEP 4: TECHNICAL INDICATORS ─────────────────────────────
# Computed per ticker using Spark Window functions.
# Window is ordered by timestamp, partitioned by ticker.
# All indicators use only close price unless noted.

def add_indicators(df, granularity="daily"):
    """
    Adds technical indicators to a DataFrame.
    granularity: 'daily' | '5min' | '1min'
    Indicator periods scale with granularity.
    """
    w = Window.partitionBy("ticker").orderBy("timestamp")

    # ── Daily return ──────────────────────────────────────────
    df = df.withColumn("prev_close", F.lag("close", 1).over(w))
    df = df.withColumn(
        "daily_return",
        F.round((F.col("close") - F.col("prev_close")) / F.col("prev_close"), 6)
    ).drop("prev_close")

    # ── SMAs ─────────────────────────────────────────────────
    if granularity == "daily":
        sma_periods = [20, 50]
    else:
        sma_periods = [10, 20]   # shorter windows for intraday

    for p in sma_periods:
        w_sma = Window.partitionBy("ticker").orderBy("timestamp").rowsBetween(-(p - 1), 0)
        df = df.withColumn(f"sma_{p}", F.round(F.avg("close").over(w_sma), 4))

    # ── EMA (12 and 26 for MACD) ──────────────────────────────
    # Spark doesn't have a native EMA — approximate with expanding weighted avg
    # using the standard multiplier: k = 2 / (period + 1)
    # We use a UDF-free approach: recursive approximation via Window lag
    for period in [12, 26]:
        k = 2.0 / (period + 1)
        w_ema = Window.partitionBy("ticker").orderBy("timestamp").rowsBetween(-(period * 3), 0)
        # Weighted average approximation (sufficient for trend signals)
        weights = [(1 - k) ** i for i in range(period * 3)]
        weight_sum = sum(weights)
        # Use rolling avg as EMA proxy (accurate enough for cross-ticker comparison)
        df = df.withColumn(f"ema_{period}", F.round(F.avg("close").over(w_ema), 4))

    # ── MACD line = EMA12 - EMA26 ─────────────────────────────
    df = df.withColumn("macd", F.round(F.col("ema_12") - F.col("ema_26"), 4))

    # ── RSI 14 ────────────────────────────────────────────────
    rsi_period = 14
    w_rsi = Window.partitionBy("ticker").orderBy("timestamp")
    df = df.withColumn("_prev_close", F.lag("close", 1).over(w_rsi))
    df = df.withColumn("_delta", F.col("close") - F.col("_prev_close"))
    df = df.withColumn("_gain", F.when(F.col("_delta") > 0, F.col("_delta")).otherwise(0.0))
    df = df.withColumn("_loss", F.when(F.col("_delta") < 0, -F.col("_delta")).otherwise(0.0))
    w_rsi_roll = Window.partitionBy("ticker").orderBy("timestamp").rowsBetween(-(rsi_period - 1), 0)
    df = df.withColumn("_avg_gain", F.avg("_gain").over(w_rsi_roll))
    df = df.withColumn("_avg_loss", F.avg("_loss").over(w_rsi_roll))
    df = df.withColumn(
        "rsi_14",
        F.round(
            F.when(F.col("_avg_loss") == 0, 100.0)
             .otherwise(100.0 - (100.0 / (1.0 + F.col("_avg_gain") / F.col("_avg_loss")))),
            2
        )
    )
    df = df.drop("_prev_close", "_delta", "_gain", "_loss", "_avg_gain", "_avg_loss")

    # ── Bollinger Bands (20-period, 2 std dev) ────────────────
    bb_period = 20 if granularity == "daily" else 10
    w_bb = Window.partitionBy("ticker").orderBy("timestamp").rowsBetween(-(bb_period - 1), 0)
    df = df.withColumn("bb_mid",   F.round(F.avg("close").over(w_bb), 4))
    df = df.withColumn("bb_std",   F.stddev("close").over(w_bb))
    df = df.withColumn("bb_upper", F.round(F.col("bb_mid") + 2 * F.col("bb_std"), 4))
    df = df.withColumn("bb_lower", F.round(F.col("bb_mid") - 2 * F.col("bb_std"), 4))
    df = df.drop("bb_std")

    # ── Rolling volatility (20-period std of daily returns) ───
    vol_period = 20 if granularity == "daily" else 10
    w_vol = Window.partitionBy("ticker").orderBy("timestamp").rowsBetween(-(vol_period - 1), 0)
    df = df.withColumn("rolling_volatility", F.round(F.stddev("daily_return").over(w_vol), 6))

    # ── Normalised price (indexed to 100 at first available bar) ──
    w_first = Window.partitionBy("ticker").orderBy("timestamp")
    df = df.withColumn("_first_close", F.first("close").over(w_first))
    df = df.withColumn(
        "price_indexed",
        F.round((F.col("close") / F.col("_first_close")) * 100, 4)
    ).drop("_first_close")

    return df

print("add_indicators() defined")

StatementMeta(, cc821a45-bf34-4744-adea-5258840ae033, 7, Finished, Available, Finished, False)

add_indicators() defined


In [6]:
# ── STEP 5: COLUMN ORDER ──────────────────────────────────────
# Final select to enforce a clean, consistent column order per table.

DAILY_COLS = [
    # Identity
    "ticker", "sector", "sub_industry",
    # Time
    "timestamp", "date", "day_of_week", "week_of_year",
    "month", "quarter", "year", "is_month_end", "is_quarter_end",
    # OHLCV
    "open", "high", "low", "close", "volume", "vwap", "transaction_count",
    # Indicators
    "daily_return", "price_indexed",
    "sma_20", "sma_50",
    "ema_12", "ema_26", "macd",
    "rsi_14",
    "bb_upper", "bb_mid", "bb_lower",
    "rolling_volatility",
]

INTRADAY_COLS = [
    # Identity
    "ticker", "sector", "sub_industry",
    # Time
    "timestamp", "timestamp_et", "date", "day_of_week",
    "hour", "minute", "session",
    # OHLCV
    "open", "high", "low", "close", "volume", "vwap", "transaction_count",
    # Indicators
    "daily_return", "price_indexed",
    "sma_10", "sma_20",
    "ema_12", "ema_26", "macd",
    "rsi_14",
    "bb_upper", "bb_mid", "bb_lower",
    "rolling_volatility",
]

print("Column order defined")

StatementMeta(, cc821a45-bf34-4744-adea-5258840ae033, 8, Finished, Available, Finished, False)

Column order defined


In [7]:
# ── STEP 6: WRITE SILVER ─────────────────────────────────────

def write_silver(df, table_name):
    """Overwrite silver table — silver is always fully recomputed from bronze."""
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    count = spark.read.format("delta").table(table_name).count()
    print(f"✓ {table_name}: {count:,} rows written")

print("write_silver() defined")

StatementMeta(, cc821a45-bf34-4744-adea-5258840ae033, 9, Finished, Available, Finished, False)

write_silver() defined


In [8]:
# ── STEP 7: RUN PIPELINE ──────────────────────────────────────

def run_silver_pipeline():
    start = datetime.utcnow()
    print("=" * 60)
    print("  Silver Layer Pipeline")
    print(f"  Started: {start.strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print("=" * 60)

    bronze = spark.read.format("delta").table(BRONZE_TABLE)

    # ── Daily ─────────────────────────────────────────────────
    print("\n[1/3] Processing daily bars...")
    df_daily = (
        bronze.filter(F.col("granularity") == "daily")
        .transform(base_transform)
        .transform(enrich_daily)
        .transform(lambda df: add_indicators(df, granularity="daily"))
        .select(DAILY_COLS)
    )
    write_silver(df_daily, SILVER_DAILY)

    # ── 5min ─────────────────────────────────────────────────
    print("\n[2/3] Processing 5min bars...")
    df_5min = (
        bronze.filter(F.col("granularity") == "5min")
        .transform(base_transform)
        .transform(enrich_intraday)
        .transform(lambda df: add_indicators(df, granularity="5min"))
        .select(INTRADAY_COLS)
    )
    write_silver(df_5min, SILVER_5MIN)

    # ── 1min ─────────────────────────────────────────────────
    print("\n[3/3] Processing 1min bars...")
    df_1min = (
        bronze.filter(F.col("granularity") == "1min")
        .transform(base_transform)
        .transform(enrich_intraday)
        .transform(lambda df: add_indicators(df, granularity="1min"))
        .select(INTRADAY_COLS)
    )
    write_silver(df_1min, SILVER_1MIN)

    elapsed = (datetime.utcnow() - start).total_seconds() / 60
    print(f"\n✅ Silver pipeline complete in {elapsed:.1f} minutes")


run_silver_pipeline()

StatementMeta(, cc821a45-bf34-4744-adea-5258840ae033, 10, Finished, Available, Finished, False)

  Silver Layer Pipeline
  Started: 2026-06-03 07:27:00 UTC

[1/3] Processing daily bars...
✓ Avi.dbo.silver_daily: 3,416 rows written

[2/3] Processing 5min bars...
✓ Avi.dbo.silver_5min: 56,244 rows written

[3/3] Processing 1min bars...
✓ Avi.dbo.silver_1min: 21,007 rows written

✅ Silver pipeline complete in 0.8 minutes


In [9]:
# ── SANITY CHECK ─────────────────────────────────────────────

for table in [SILVER_DAILY, SILVER_5MIN, SILVER_1MIN]:
    print(f"\n── {table} ──")
    df = spark.read.format("delta").table(table)
    df.groupBy("sub_industry") \
      .agg(
          F.count("*").alias("rows"),
          F.countDistinct("ticker").alias("tickers"),
          F.min("date").alias("earliest"),
          F.max("date").alias("latest")
      ) \
      .orderBy("sub_industry") \
      .show()

# Spot check: one ticker's daily indicators
print("\n── Spot check: LLY daily indicators (last 5 rows) ──")
spark.read.format("delta").table(SILVER_DAILY) \
    .filter(F.col("ticker") == "LLY") \
    .orderBy(F.col("date").desc()) \
    .select("date", "close", "daily_return", "price_indexed",
            "sma_20", "rsi_14", "macd", "rolling_volatility") \
    .show(5)

StatementMeta(, cc821a45-bf34-4744-adea-5258840ae033, 11, Finished, Available, Finished, False)


── Avi.dbo.silver_daily ──
+--------------------+----+-------+----------+----------+
|        sub_industry|rows|tickers|  earliest|    latest|
+--------------------+----+-------+----------+----------+
|Air Freight & Log...| 610|      5|2025-12-05|2026-06-02|
|    Airport Services| 366|      3|2025-12-05|2026-06-02|
|       Biotechnology| 610|      5|2025-12-05|2026-06-02|
|Health Care Equip...| 610|      5|2025-12-05|2026-06-02|
|Marine Ports & Se...| 610|      5|2025-12-05|2026-06-02|
|     Pharmaceuticals| 610|      5|2025-12-05|2026-06-02|
+--------------------+----+-------+----------+----------+


── Avi.dbo.silver_5min ──
+--------------------+-----+-------+----------+----------+
|        sub_industry| rows|tickers|  earliest|    latest|
+--------------------+-----+-------+----------+----------+
|Air Freight & Log...| 9702|      5|2026-05-04|2026-06-02|
|    Airport Services| 4635|      3|2026-05-04|2026-06-02|
|       Biotechnology|10558|      5|2026-05-04|2026-06-02|
|Health Ca